# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # mlcroissant DatasetMetadata object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect the available record sets and their fields. All references are by their `@id` as required.

In [ ]:
record_sets = dataset.metadata.record_sets

print("Available Record Sets:\n")
for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}\n")

    print("  Fields:")
    for field in rs.fields:
        print(f"    @id: {field.id}, name: {field.name}, type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

The following extracts all tabular record sets and loads them into `pandas` DataFrames.

In [ ]:
# Gather all record set IDs
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  [!] No records found for {record_set_id}. Skipping.\n")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  --> Columns: {df.columns.tolist()}\n")

# Let's pick the first available dataframe for demonstration
main_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"Using RecordSet: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No available records loaded into dataframes.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick a numeric field from the selected record set
# You may update this field depending on inspected columns

df = dataframes[main_record_set_id]

# Guess a numeric field (update as appropriate; here are some reasonable guesses):
possible_numeric_fields = [c for c in df.columns if (
    ('abundance' in c.lower() or 'count' in c.lower() or 'coverage' in c.lower() or 'weight' in c.lower() or 'mw' in c.lower())
    and pd.api.types.is_numeric_dtype(df[c])
)]
# If none found, pick the first numeric column
if not possible_numeric_fields:
    possible_numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try to find a suitable group field
    possible_group_fields = [c for c in df.columns if any(k in c.lower() for k in ('group', 'sample', 'type', 'class', 'modif', 'category'))]
    group_field = possible_group_fields[0] if possible_group_fields else None
    
    if group_field:
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(grouped_df.head())
else:
    print("No numeric field found for analysis in this RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Adjust field names as appropriate to your dataset contents.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and possible_numeric_fields:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a group field, plot boxplot
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("Visualization skipped: No suitable numeric or categorical field found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded a Croissant-structured dataset describing proteins measured by mass spectrometry from human mast cell extracellular vesicles.
- We explored the record set and fields available, loaded a main table, and demonstrated basic EDA and filtering using the `mlcroissant` APIs and `pandas`.
- For deeper analysis, consult the dataset's documentation, field annotations, and scientific context.